# Lab L4.3 · Neural-network tuning clinic — learning rate, batch size, over-fitting, imbalance and embeddings on churn
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 4 · ⏱ 75 min · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Challenge the Day 3 churn champion with a Keras MLP, and learn by experiment what each training knob does — the practical skills behind every deep-learning project.

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the 💡 hint in the HTML lab guide, then ask your trainer. The Level 3 solution is released after the lab.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

The Head of Retention read that ‘deep learning beats everything’ and asks MIZAN to try a neural network on churn. Rather than one lucky run, you run a **tuning clinic**: a sequence of controlled experiments that shows the team what each hyperparameter does, ending in a fair **champion vs challenger** decision on the *same* test set as L3.4.

**Progressive architecture:** reuses the L3.4 split and fitted preprocessing so the comparison is exact; the decision feeds the Day 5 capstone (L5.3).

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Load the champion's split and preprocessing

**Why this matters.** A challenger is only comparable if it sees the **same rows** and the **same features**. We reload the L3.4 split ids and reuse the champion's fitted `ColumnTransformer`, then convert to dense `float32` arrays — the format Keras expects.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(RNG)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

In [ ]:
import json
cust = pd.read_parquet(CKPT / "customers_clean.parquet")                    # L1.1
seg = pd.read_parquet(CKPT / "customer_segments.parquet")                  # L3.2
txf = pd.read_parquet(CKPT / "customer_txn_features.parquet")              # L1.3
TXN = ["has_card_activity", "txn_count", "log_amt_mean", "recency_days", "spend_trend", "ecom_share", "n_mcc"]
df = (cust.merge(seg[["customer_id", "segment"]], on="customer_id", validate="one_to_one")
          .merge(txf[["customer_id"] + TXN], on="customer_id", validate="one_to_one"))
df["log_balance"], df["log_salary"] = np.log1p(df.avg_balance_aed), np.log1p(df.monthly_salary_aed)
NUM = ["age", "tenure_months", "num_products", "salary_transfer", "has_credit_card", "has_personal_loan", "has_auto_loan", "has_mortgage",
       "log_balance", "log_salary", "digital_logins_30d", "branch_visits_90d", "complaints_12m", "days_since_last_txn", "aecb_score"] + TXN
CAT = ["segment", "employment_type", "emirate"]
FORBIDDEN = {"account_closed_date", "retention_offer_made", "gender", "nationality_group"}   # leakage + protected attributes
import joblib
from sklearn.metrics import roc_auc_score, average_precision_score
champ = joblib.load(CKPT / "churn_model.joblib")                            # L3.4 champion + its split
ids = champ["split_ids"]
tr, va, te = (df[df.customer_id.isin(ids[s])] for s in ["train", "valid", "test"])
pre = champ["model"][:-1]                                                   # the champion's fitted preprocessing
dense = lambda Z: np.asarray(Z.todense() if hasattr(Z, "todense") else Z, dtype="float32")
Xtr, Xva, Xte = (dense(pre.transform(d[NUM + CAT])) for d in (tr, va, te))
ytr, yva, yte = (d.churned_6m.to_numpy() for d in (tr, va, te))
print(f"{Xtr.shape[1]} input features | train {len(ytr):,} · valid {len(yva):,} · test {len(yte):,} | champion: {champ['name']} "
      f"(test PR-AUC {champ['test']['PR_AUC']:.3f})")

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — non-linear activation for hidden layers
- **Blank 2** *(inline)* — squash to a probability for a binary target

In [ ]:
def make_mlp(units=(64, 32), lr=1e-3, dropout=0.0, l2=0.0, n_in=Xtr.shape[1]):
    reg = keras.regularizers.l2(l2) if l2 else None
    m = keras.Sequential([keras.Input((n_in,))])
    for u in units:
        m.add(layers.Dense(u, activation=____, kernel_regularizer=reg))  # ✏️ B1
        if dropout: m.add(layers.Dropout(dropout))
    m.add(layers.Dense(1, activation=____))  # ✏️ B2
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
              metrics=[keras.metrics.AUC(name="auc"), keras.metrics.AUC(curve="PR", name="pr_auc")])
    return m
make_mlp().summary()

**✔ Checkpoint.** 37 input features; 150k / 50k / 50k rows; champion = logistic (test PR-AUC ≈ 0.25). The default MLP has 4,545 parameters.

## Step 2 · Experiment 1 — the learning rate

**Why this matters.** The learning rate is the single most important optimiser setting. Too small: the loss creeps down and you run out of epochs. Too large: updates overshoot and the loss bounces or explodes. The **validation loss curve** shows which regime you are in within a few epochs.

#### ✏️ What to write in the next cell
- **Blank 3** *(inline)* — same data, only lr changes

In [ ]:
lr_res, curves = {}, {}
for lr in [1e-5, 1e-3, 1e-2, 0.3]:
    keras.utils.set_random_seed(RNG)
    h = ____  # ✏️ B3
    curves[lr] = h.history["val_loss"]; lr_res[lr] = max(h.history["val_pr_auc"])
for lr, c in curves.items(): plt.plot(range(1, 9), c, marker="o", label=f"lr={lr}")
plt.xlabel("epoch"); plt.ylabel("validation loss"); plt.legend(); plt.title("Learning-rate experiment"); plt.show()
pd.Series(lr_res, name="best val PR-AUC").round(4).to_frame()

**✔ Checkpoint.** lr 1e-5 is still far from converged after 8 epochs (PR-AUC ≈ 0.14); 1e-3 and 1e-2 converge to ≈ 0.25; 0.3 gives an erratic, higher loss curve and PR-AUC ≈ 0.19.

## Step 3 · Experiment 2 — batch size, updates and time

**Why this matters.** One **epoch** = one pass over the data; the number of **updates** per epoch = rows ÷ batch size. Small batches give many noisy updates (slow per epoch, good at escaping flat regions); large batches give few smooth updates (fast on hardware, may need a larger learning rate). We measure both quality and seconds.

#### ✏️ What to write in the next cell
- **Blank 4** *(inline)* — rows ÷ batch size

In [ ]:
rows = []
for batch, lr in [(32, 1e-3), (512, 1e-3), (8192, 1e-3), (8192, 1e-2)]:
    keras.utils.set_random_seed(RNG); t0 = time.time()
    h = make_mlp(lr=lr).fit(Xtr, ytr, validation_data=(Xva, yva), epochs=3, batch_size=batch, verbose=0)
    rows.append({"batch": batch, "lr": lr, "updates_per_epoch": ____,  # ✏️ B4
                 "seconds": time.time() - t0, "best_val_pr_auc": max(h.history["val_pr_auc"])})
pd.DataFrame(rows).round(4)

**✔ Checkpoint.** Batch 32: 4,688 updates/epoch, ≈ 7× slower than batch 512, converged (≈ 0.25). Batch 8192 with lr 1e-3: only 19 updates/epoch — fastest but under-trained (≈ 0.20); raising lr to 1e-2 recovers most of the gap (≈ 0.24).

## Step 4 · Experiment 3 — provoke over-fitting, then cure it

**Why this matters.** With few rows and many parameters a network memorises noise: training loss keeps falling while validation loss turns upward. The cures are the regularisers you met on Day 2 in neural form: **dropout** (randomly silence neurons during training), **L2 weight decay**, and **early stopping** (keep the weights from the best validation epoch).

#### ✏️ What to write in the next cell
- **Blank 5** *(inline)* — stop when validation loss stops improving
- **Blank 6** *(inline)* — silence 40% of neurons at each training step

In [ ]:
small = np.random.default_rng(RNG).choice(len(ytr), 4_000, replace=False)
def run(**kw):
    keras.utils.set_random_seed(RNG)
    cb = kw.pop("callbacks", [])
    m = make_mlp((256, 256), **kw)
    h = m.fit(Xtr[small], ytr[small], validation_data=(Xva, yva), epochs=60, batch_size=128, verbose=0, callbacks=cb)
    return m, h.history
m_over, h_over = run(lr=1e-3)
stop = ____  # ✏️ B5
m_reg, h_reg = run(lr=1e-3, dropout=____, l2=1e-3, callbacks=[stop])  # ✏️ B6
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for a, h, t in [(ax[0], h_over, "no regularisation"), (ax[1], h_reg, "dropout + L2 + early stopping")]:
    a.plot(h["loss"], label="train"); a.plot(h["val_loss"], label="validation"); a.set_title(t); a.set_xlabel("epoch"); a.legend()
ax[0].set_ylabel("loss"); ax[0].set_ylim(0, 0.8); plt.show()
ev = lambda m: average_precision_score(yva, m.predict(Xva, verbose=0).ravel())
print(f"val PR-AUC — over-fitted: {ev(m_over):.3f} | regularised: {ev(m_reg):.3f} (stopped after {len(h_reg['loss'])} epochs)")

**✔ Checkpoint.** Unregularised: training loss heads towards 0 while validation loss climbs after a few epochs; val PR-AUC ≈ 0.17. Regularised: the curves stay together, training stops after ≈ 25 epochs and val PR-AUC recovers to ≈ 0.23 — from only 4,000 rows.

## Step 5 · Experiment 4 — class weights: ranking vs calibration

**Why this matters.** With 12% churners, `class_weight={0: 1, 1: 7.3}` makes each churner count as much as all non-churners together. It often changes **ranking** very little (PR-AUC) but pushes every probability up — the model is no longer **calibrated**. Retention EV maths (L3.4) needs calibrated probabilities.

#### ✏️ What to write in the next cell
- **Blank 7** *(inline)* — each churner counts ≈ 7× more

In [ ]:
w1 = (1 - ytr.mean()) / ytr.mean()
res_cw = {}
for name, cw in [("no weights", None), ("balanced weights", ____)]:  # ✏️ B7
    keras.utils.set_random_seed(RNG)
    m = make_mlp().fit(Xtr, ytr, epochs=6, batch_size=512, class_weight=cw, verbose=0).model
    p = m.predict(Xva, verbose=0).ravel()
    res_cw[name] = {"val_PR_AUC": average_precision_score(yva, p), "mean_pred": p.mean(), "actual_rate": yva.mean()}
pd.DataFrame(res_cw).T.round(3)

**✔ Checkpoint.** PR-AUC is identical (≈ 0.25), but the weighted model's mean prediction is ≈ 0.45 against an actual rate of 0.12 — badly mis-calibrated. Unweighted: 0.12, spot on.

## Step 6 · Experiment 5 — embeddings instead of one-hot

**Why this matters.** One-hot encoding gives every category its own independent column. An **embedding** layer learns a small dense vector per category, so similar categories (e.g. emirates with similar churn behaviour) end up close together — the same idea that powers word embeddings. It needs the **functional API** because the model has several inputs.

#### ✏️ What to write in the next cell
- **Blank 8** *(inline)* — learn a 3-number vector per category

In [ ]:
num_pre = pre.named_steps["columntransformer"].named_transformers_["num"]   # imputer + scaler fitted in L3.4
codes = {c: pd.Categorical(df[c]).categories for c in CAT}
def inputs(d):
    feats = {"num": num_pre.transform(d[NUM]).astype("float32")}
    feats.update({c: pd.Categorical(d[c], categories=codes[c]).codes.astype("int32") for c in CAT})
    return feats
Itr, Iva, Ite = inputs(tr), inputs(va), inputs(te)
num_in = keras.Input((len(NUM),), name="num")
cat_in = [keras.Input((1,), name=c, dtype="int32") for c in CAT]
emb = [layers.Flatten()(____)  # ✏️ B8
       for c, i in zip(CAT, cat_in)]
x = layers.Concatenate()([num_in, *emb])
x = layers.Dense(64, activation="relu")(x); x = layers.Dense(32, activation="relu")(x)
emb_model = keras.Model([num_in, *cat_in], layers.Dense(1, activation="sigmoid")(x))
emb_model.compile(keras.optimizers.Adam(1e-3), "binary_crossentropy", metrics=[keras.metrics.AUC(curve="PR", name="pr_auc")])
keras.utils.set_random_seed(RNG)
h = emb_model.fit(Itr, ytr, validation_data=(Iva, yva), epochs=8, batch_size=512, verbose=0)
print(f"embedding model best val PR-AUC: {max(h.history['val_pr_auc']):.4f} | one-hot MLP (lr 1e-3): {lr_res[1e-3]:.4f}")
pd.DataFrame(emb_model.get_layer("emb_emirate").get_weights()[0], index=codes["emirate"]).round(2)

**✔ Checkpoint.** Embeddings match the one-hot MLP (within ≈ 0.005 PR-AUC) — with only 7 emirates there is little to gain; embeddings shine with hundreds of categories (merchants, branches, products). The 7 learned emirate vectors are printed.

## Step 7 · Final challenger vs champion on the test set

**Why this matters.** Now combine what worked — lr 1e-3, batch 512, moderate dropout, early stopping on validation — train on the full training set and score the **test** set once, exactly as the champion was. A challenger replaces the champion only if it is *materially* better (here: ≥ 0.01 PR-AUC), because a neural network costs more to run, explain and govern.

#### ✏️ What to write in the next cell
- **Blank 9** *(inline)* — material improvement threshold

In [ ]:
import json
keras.utils.set_random_seed(RNG)
best = make_mlp((128, 64), lr=1e-3, dropout=0.2)
best.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=40, batch_size=512, verbose=0,
         callbacks=[keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=4, restore_best_weights=True)])
p = best.predict(Xte, verbose=0).ravel()
top = np.argsort(-p)[: len(p) // 10]
mlp_test = {"ROC_AUC": roc_auc_score(yte, p), "PR_AUC": average_precision_score(yte, p), "capture@10%": float(yte[top].sum() / yte.sum())}
cmp = pd.DataFrame({f"champion ({champ['name']})": champ["test"], "MLP challenger": mlp_test}).T
display(cmp.round(4))
gain = mlp_test["PR_AUC"] - champ["test"]["PR_AUC"]
decision = ____  # ✏️ B9
print(f"PR-AUC gain {gain:+.4f} → {decision}")
best.save(CKPT / "churn_mlp.keras")
_ = (CKPT / "nn_tuning_results.json").write_text(json.dumps({"lr_sweep_val_pr_auc": {str(k): float(v) for k, v in lr_res.items()},
    "class_weights": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in res_cw.items()},
    "test": {"champion": champ["test"], "mlp": mlp_test}, "decision": decision}, indent=2, default=float))

**✔ Checkpoint.** The MLP lands slightly *below* the logistic champion (PR-AUC ≈ 0.247 vs 0.251) → **KEEP champion**. Deep learning does not win on small, tabular, mostly-linear problems; it wins on images (L4.2's CNN: 97% vs 69%) and sequences.

## ✅ Lab complete
You can now tune a network deliberately and defend a *no*. **Tomorrow:** shipping and monitoring the churn model in the capstone.